# Lost in Transcription — Spanish–English

Independent Spanish–English workflow. Do not mix its data, weights, glossary, TensorBoard logs, Hub repository, or submission archive with the Indonesian–Javanese or Spanish–Nahuatl tracks.

In [ ]:
!pip install -r requirements-dev.txt

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from getpass import getpass

def find_project_root() -> Path:
    required = ("requirements-runpod.txt", "src", "spanish_english")
    configured = os.environ.get('LIT_ROOT')
    candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents]

    for candidate in candidates:
        if all((candidate / item).exists() for item in required):
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate the project root. Open the notebook from the repository "
        "or set ROOT manually."
    )

ROOT = find_project_root()

if not ROOT.is_dir():
    raise FileNotFoundError(f'Update ROOT: {ROOT}')
# Installs requirements-runpod.txt, repairing any apt-managed package without a pip RECORD file
# (blinker, pyparsing, ...), then runs scripts/verify_runpod.py.
subprocess.run([sys.executable, str(ROOT / 'scripts' / 'install_runpod.py')], check=True)
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
SPEN_DATA = ROOT / 'data' / 'spanish_english'
SPEN_DEV_DIR = SPEN_DATA / 'development'
BANGOR_DIR = SPEN_DATA / 'bangor_miami'
ARTIFACTS = ROOT / 'artifacts' / 'spen'
TENSORBOARD_ROOT = ARTIFACTS / 'tensorboard'
ARTIFACTS.mkdir(parents=True, exist_ok=True); TENSORBOARD_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Downloads only when Bangor is absent; the script reads MDC_API_KEY from the environment or ROOT / '.env'.
if not (BANGOR_DIR.exists() and any(p.is_file() and not p.name.startswith('.') for p in BANGOR_DIR.rglob('*'))) and not (os.environ.get('MDC_API_KEY') or (ROOT / '.env').is_file()):
    os.environ['MDC_API_KEY'] = getpass('MDC API key (not saved): ')
if not (BANGOR_DIR.exists() and any(p.is_file() and not p.name.startswith('.') for p in BANGOR_DIR.rglob('*'))):
    subprocess.run([sys.executable, str(ROOT / 'spanish_english' / 'download_bangor_miami.py'), '--download'], check=True, env=os.environ.copy())
else:
    print(f'Reusing {BANGOR_DIR}')

In [ ]:
from spanish_english import prepare
from lit_jv import prompts
from lit_jv.decode import DecodeConfig, FallbackConfig, Qwen3AsrDecoder, build_backend
from lit_jv.scoring import wer_report
from lit_jv.train_lora import AugmentSpec, LoraSpec, StageSpec, TrainSpec, merge_and_save, train

dev_rows = prepare.load_competition_dev(SPEN_DEV_DIR)
bangor_rows = prepare.load_bangor_miami(BANGOR_DIR)
dev_train, dev_eval = prepare.holdout_speakers(dev_rows, count=1, seed=2026)
selection_train = bangor_rows + dev_train
terms = prepare.glossary(selection_train)
(ARTIFACTS / 'glossary.json').write_text(json.dumps(terms, ensure_ascii=False, indent=2), encoding='utf-8')
print({'bangor_train': len(bangor_rows), 'dev_train': len(dev_train), 'dev_eval': len(dev_eval), 'glossary': len(terms)})

prompt = prompts.PromptConfig(prose='codeswitch_spen', style_tag=prompts.STYLE_VOICE_NOTE_SP_EN, glossary=tuple(terms), max_glossary_terms=128, force_language='Spanish')
def score(model_path, rows, name):
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(backend, fallback=FallbackConfig(enabled=False), log=print)
    try:
        hypotheses = decoder.decode_pass(rows, DecodeConfig(prompt=prompt, name=name, temperature=0.0), chunk_size=16)
        return wer_report([row['raw_text'] for row in rows], [hyp.text for hyp in hypotheses])
    finally:
        decoder.close()
baseline = score('Qwen/Qwen3-ASR-1.7B', dev_eval, 'qwen-zero-shot')
print(baseline.as_row('qwen-zero-shot'))

## 4.5 Multi-pass prompts, error taxonomy, and post-processing

Ported from the Indonesian-Javanese track, which found this combination worth several WER
points before any acoustic change: (1) decode a few deliberately decorrelated prompt
variants instead of one, so ROVER/MBR later has something real to vote over; (2) turn the
truncation-rescue fallback back on (`score_rows` above ships it disabled); (3) fit a
forward-selected post-processing pipeline that only keeps a rule if it lowers WER.

Caveat worth stating plainly: this dev set gives one held-out-speaker split, not the
multi-conversation fold the JV track validates its rules against, so there is no
independent check available beyond `dev_eval` itself here. Treat the rules below as
provisional until §5.5 re-fits them on the fine-tuned model's own hypotheses.

In [ ]:
from lit_jv import combine, postprocess
from collections import Counter

# --- multi-pass prompt design -----------------------------------------
PASS_SPECS = [
    ("p1-glossary", prompt, 0.0),
    (
        "p2-noglossary",
        prompts.PromptConfig(prose='codeswitch_spen', style_tag=prompts.STYLE_VOICE_NOTE_SP_EN, glossary=(),
                              force_language='Spanish'),
        0.0,
    ),
    (
        "p3-freelid",
        prompts.PromptConfig(prose='codeswitch_spen', style_tag=prompts.STYLE_VOICE_NOTE_SP_EN, glossary=tuple(terms[:32]),
                              force_language=None),
        0.0,
    ),
]

# --- length prior + fallback-enabled decode helper ----------------------
HAVE_DURATIONS = bool(dev_eval) and dev_eval[0].get('duration') is not None
if HAVE_DURATIONS:
    PRIOR = combine.LengthPrior.fit(
        [r['raw_text'] for r in dev_train], [r['duration'] for r in dev_train]
    )
    print(f"length prior: mean={PRIOR.mean:.3f} std={PRIOR.std:.3f} words/s")
else:
    PRIOR = None
    print("no 'duration' field on these rows -- fallback ladder / length-prior gating "
          "skipped. Check the field name in prepare.py if that's unexpected.")


def decode_multi_pass(model_path, rows, specs=PASS_SPECS, tag='pass'):
    """Decode `rows` under every spec in `specs`. Returns {name: [text, ...]}."""
    backend = build_backend(model_path, prefer='vllm', log=print)
    decoder = Qwen3AsrDecoder(
        backend, fallback=FallbackConfig(enabled=HAVE_DURATIONS), length_prior=PRIOR, log=print
    )
    out = {}
    try:
        for name, cfg, temperature in specs:
            hyps = decoder.decode_pass(
                rows, DecodeConfig(prompt=cfg, name=f'{tag}-{name}', temperature=temperature),
                chunk_size=16,
            )
            out[name] = [h.text for h in hyps]
    finally:
        decoder.close()
    return out


# --- decode every pass zero-shot, pick the best single one --------------
ZERO_SHOT_PASSES = decode_multi_pass('Qwen/Qwen3-ASR-1.7B', dev_eval, tag='zeroshot')
refs = [r['raw_text'] for r in dev_eval]
zero_shot_wer = {name: round(wer_report(refs, texts).wer, 4) for name, texts in ZERO_SHOT_PASSES.items()}
for name, wer in sorted(zero_shot_wer.items(), key=lambda kv: kv[1]):
    print(f"{name:16s} WER {wer}")
ZS_BEST_PASS = min(zero_shot_wer, key=zero_shot_wer.get)
print('best zero-shot pass:', ZS_BEST_PASS)

# --- error taxonomy on the best pass -------------------------------------
def classify(ref_token, hyp_token):
    if not ref_token:
        return 'insertion'
    if not hyp_token:
        return 'deletion'
    if ref_token == hyp_token:
        return 'identical (alignment artefact)'
    if ref_token.lower() == hyp_token.lower():
        return 'casing only'
    strip = postprocess.strip_accents
    if strip(ref_token).lower() == strip(hyp_token).lower():
        return 'accent only'
    if ref_token.replace('-', '') == hyp_token.replace('-', ''):
        return 'hyphen'
    if strip(ref_token.replace('-', '')).lower() == strip(hyp_token.replace('-', '')).lower():
        return 'mixed style (casing+accent+hyphen)'
    return 'lexical / acoustic'

pairs = [
    (rt, ht)
    for ref, hyp in zip(refs, ZERO_SHOT_PASSES[ZS_BEST_PASS])
    for rt, ht in zip(ref.split(), hyp.split())
]
tax = Counter(classify(a, b) for a, b in pairs if a != b)
total = sum(tax.values()) or 1
for k, v in tax.most_common():
    print(f"{k:35s} {v:5d}  ({v / total:.1%})")

# --- forward-selected post-processing, gated on WER ----------------------
LEXICON = postprocess.StyleLexicon.fit(refs, min_count=2, dominance=0.9)
PIPELINE = postprocess.RulePipeline(lexicon=LEXICON).fit(
    refs, ZERO_SHOT_PASSES[ZS_BEST_PASS], verbose=True
)
_durations = [r['duration'] for r in dev_eval] if HAVE_DURATIONS else [0.0] * len(dev_eval)
_repaired = PIPELINE.apply_many(ZERO_SHOT_PASSES[ZS_BEST_PASS], _durations)
print(f"{ZS_BEST_PASS} + post-processing: {wer_report(refs, _repaired).wer:.4f} "
      f"(zero-shot alone: {zero_shot_wer[ZS_BEST_PASS]:.4f})")


In [ ]:
spec = TrainSpec(
    model_path='Qwen/Qwen3-ASR-1.7B', output_dir=str(ARTIFACTS / 'lora'), target_language='Spanish',
    lora=LoraSpec(r=16, alpha=32, adapt_text_decoder=True, adapt_audio_encoder=False),
    augment=AugmentSpec(enabled=False), prompt=prompt, eval_steps=0, wer_eval_steps=10,
    wer_eval_clips=len(dev_eval), wer_eval_batch_size=4, tensorboard_dir=str(TENSORBOARD_ROOT / 'selection'),
    report_to=('tensorboard',), run_name='spen-qwen3asr',
)
stages = (StageSpec('s1-bangor', {'bangor_miami': 1.0}, epochs=1.0, learning_rate=8e-5, batch_size=4, grad_accumulation=8), StageSpec('s2-dev', {'spen_dev': 1.0}, epochs=3.0, learning_rate=3e-5, batch_size=4, grad_accumulation=8))
selected = train(selection_train, spec, stages=stages, save_adapter_as='selected', eval_rows=dev_eval)
merged = merge_and_save(spec, ARTIFACTS / 'lora' / 'selected', ARTIFACTS / 'merged')
selected_dev = score(merged, dev_eval, 'selected-dev')
assert selected_dev.wer < baseline.wer, 'Reject: selected adapter did not beat zero-shot.'
print(selected_dev.as_row('selected-dev'))


### 5.5 Combine passes on the selection checkpoint (held-out, evidence-based)

Decode `dev_eval` with the **selection** checkpoint (before the final fit rolls it into full
training data) across all three prompt passes, so the pass-count / combiner decision is
measured on this model rather than assumed from the zero-shot correlation. This mirrors the
JV track's own finding that ROVER/MBR are unreliable with only two candidates (a 1-1 tie
resolved by tie-break) — so `none` is a legitimate winner here, not a fallback of last
resort.

In [ ]:
SEL_PASSES = decode_multi_pass(merged, dev_eval, tag='selected')
sel_wer = {name: round(wer_report(refs, texts).wer, 4) for name, texts in SEL_PASSES.items()}
for name, wer in sorted(sel_wer.items(), key=lambda kv: kv[1]):
    print(f"{name:16s} WER {wer}")
SEL_BEST_PASS = min(sel_wer, key=sel_wer.get)

# Refit post-processing on this checkpoint's own mistakes.
SEL_LEXICON = postprocess.StyleLexicon.fit(refs, min_count=2, dominance=0.9)
SEL_PIPELINE = postprocess.RulePipeline(lexicon=SEL_LEXICON).fit(
    refs, SEL_PASSES[SEL_BEST_PASS], verbose=True
)

# Measure ROVER / MBR at every surviving pass count -- do not assume voting helps.
combiner_by_n = {}
sel_names = list(SEL_PASSES)
for n in range(1, len(sel_names) + 1):
    subset = sel_names[:n]
    single = wer_report(refs, SEL_PIPELINE.apply_many(SEL_PASSES[subset[0]], _durations)).wer
    best_label, best_wer = "none", single
    if n >= 2:
        for label, fn in (("mbr", combine.mbr_select), ("rover", combine.rover_combine)):
            texts = []
            for i in range(len(dev_eval)):
                candidates = [SEL_PASSES[m][i] for m in subset]
                texts.append(fn(candidates).text)
            wer = wer_report(refs, SEL_PIPELINE.apply_many(texts, _durations)).wer
            if wer < best_wer:
                best_label, best_wer = label, wer
    combiner_by_n[n] = best_label
    print(f"{n} pass(es): single={single:.4f}  selected={best_label} ({best_wer:.4f})")

SUBMISSION_COMBINER = combiner_by_n[len(sel_names)]
if SUBMISSION_COMBINER == "none":
    SUBMISSION_SPECS = [next(s for s in PASS_SPECS if s[0] == SEL_BEST_PASS)]
    print(f"shipping a single pass ({SEL_BEST_PASS}); combining did not help on this checkpoint.")
else:
    SUBMISSION_SPECS = PASS_SPECS
    print(f"shipping all {len(sel_names)} passes with combiner={SUBMISSION_COMBINER!r}.")


In [ ]:
final_spec = TrainSpec(**{**spec.__dict__, 'output_dir': str(ARTIFACTS / 'final-lora'), 'tensorboard_dir': str(TENSORBOARD_ROOT / 'final'), 'wer_eval_steps': 0})
final_run = train(bangor_rows + dev_rows, final_spec, stages=stages, save_adapter_as='final', eval_rows=None)
final_merged = merge_and_save(final_spec, ARTIFACTS / 'final-lora' / 'final', ARTIFACTS / 'final-merged')
print('TensorBoard:', TENSORBOARD_ROOT)
print('Jupyter: %load_ext tensorboard; %tensorboard --logdir ' + str(TENSORBOARD_ROOT))

### 6.5 Refit post-processing on the final checkpoint, then package

`final_merged` is trained on *all* released dev rows, so `dev_eval` is no longer held out from
it -- this refit is in-sample by construction, like the JV track's final pipeline. The held-out
check that matters already happened in §5.5 on the selection checkpoint; this step only makes the
shipped rules match the production model's own errors.

The cell then packages `submission.zip` with everything §5.5 measured: `postprocess.json` (+ its
lexicon), the selected pass list, the combiner (`combiner_by_n` for when the clock cuts a pass),
and this track's length prior. `package_submission.py` fails if any of it is missing from the ZIP.

In [ ]:
FINAL_PASSES = decode_multi_pass(final_merged, dev_rows, specs=SUBMISSION_SPECS, tag='final')
final_refs = [r['raw_text'] for r in dev_rows]
final_lexicon = postprocess.StyleLexicon.fit(final_refs, min_count=2, dominance=0.9)
FINAL_PIPELINE = postprocess.RulePipeline(lexicon=final_lexicon).fit(
    final_refs, FINAL_PASSES[SUBMISSION_SPECS[0][0]], verbose=True
)
FINAL_PIPELINE.save(ARTIFACTS / 'postprocess.json')
print(f"saved {len(FINAL_PIPELINE.rules)} refit rules to {ARTIFACTS / 'postprocess.json'}")

# --- ship exactly what was measured: passes, combiner, length prior, rules ----
def pass_config(name, cfg, temperature):
    return {
        "name": name, "prose": cfg.prose, "use_language_field": cfg.use_language_field,
        "style_tag": cfg.style_tag, "include_style_tag": cfg.include_style_tag,
        "glossary": bool(cfg.glossary), "max_glossary_terms": min(len(cfg.glossary), cfg.max_glossary_terms) if cfg.glossary else 0,
        "force_language": cfg.force_language, "temperature": temperature,
    }

# main.py injects the first `max_glossary_terms` of this list, so every pass's glossary must be a prefix of it.
assert all(tuple(cfg.glossary) == tuple(terms[:len(cfg.glossary)]) for _, cfg, _ in SUBMISSION_SPECS)
DECODE_CONFIG = {
    "passes": [pass_config(*s) for s in SUBMISSION_SPECS],
    "combiner": SUBMISSION_COMBINER,
    "combiner_by_n": {str(n): label for n, label in combiner_by_n.items() if n <= len(SUBMISSION_SPECS)},
}
if PRIOR is not None:
    DECODE_CONFIG["length_prior"] = {"mean": PRIOR.mean, "std": PRIOR.std}
(ARTIFACTS / 'decode_config.json').write_text(json.dumps(DECODE_CONFIG, indent=2, ensure_ascii=False), encoding='utf-8')

package = subprocess.run([
    sys.executable, str(ROOT / 'spanish_english' / 'package_submission.py'),
    '--model', str(final_merged), '--glossary', str(ARTIFACTS / 'glossary.json'),
    '--postprocess', str(ARTIFACTS / 'postprocess.json'), '--decode-config', str(ARTIFACTS / 'decode_config.json'),
], check=True, capture_output=True, text=True)
print(package.stdout)
SUBMISSION_ZIP = ARTIFACTS / 'submission.zip'
assert SUBMISSION_ZIP.is_file() and SUBMISSION_ZIP.stat().st_size > 0
print(f'Ready to publish: {final_merged} and {SUBMISSION_ZIP}')

## 7. Publish to Hugging Face: weights, adapters, TensorBoard, `submission.zip`

One repository per track (`SPEN_HF_REPO` overrides the default `<account>/lost-in-transcription-spanish-english`). It receives:

| path | content |
|---|---|
| `merged/` | final merged Qwen3-ASR weights (the ones inside the ZIP) |
| `adapters/selected`, `adapters/final` | the LoRA adapters (small; reproducible from the base model) |
| `runs/` | TensorBoard event files for the selection and final fits -- rendered under the repo's **Training metrics** tab |
| `submission.zip`, `submission_config.json`, `postprocess*.json`, `glossary.json` | the exact competition archive and its configuration |
| `run_manifest.json`, `README.md` | provenance (seed, data hash, versions, commit) and a model card with the WER table |

The repo is public unless `SPEN_HF_PRIVATE=1`; the rules require a winning solution to be verifiable.
`HF_TOKEN` is requested privately if it is not already a RunPod secret and is never written to an artifact.

In [ ]:
from lit_jv import hub

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass('Hugging Face write token (not saved): ')
if not os.environ['HF_TOKEN'].strip():
    raise RuntimeError('HF_TOKEN is required to publish the model and submission archive')
from huggingface_hub import HfApi
account = HfApi(token=os.environ['HF_TOKEN']).whoami()['name']

HUB = hub.HubSpec(
    repo_id=os.environ.get('SPEN_HF_REPO', f'{account}/lost-in-transcription-spanish-english'),
    private=os.environ.get('SPEN_HF_PRIVATE', '0') == '1',
    upload_merged=True,
    languages=('es', 'en'),
    tags=('automatic-speech-recognition', 'code-switching', 'spanish', 'english', 'lora', 'qwen3-asr', 'tensorboard'),
    summary='code-switched Spanish-English conversational speech, trained for the Lost in Transcription Spanish-English track',
)
STAGING = SUBMISSION_ZIP.with_suffix('')
published = hub.publish_run(
    HUB,
    adapters={'selected': ARTIFACTS / 'lora' / 'selected', 'final': ARTIFACTS / 'final-lora' / 'final'},
    merged_dir=final_merged,
    tensorboard_dir=TENSORBOARD_ROOT,
    manifest_path=final_run['manifest_path'],
    extra_files={
        'submission.zip': SUBMISSION_ZIP,
        'submission_config.json': STAGING / 'submission_config.json',
        'postprocess.json': ARTIFACTS / 'postprocess.json',
        'postprocess.lexicon.json': ARTIFACTS / 'postprocess.lexicon.json',
        'glossary.json': ARTIFACTS / 'glossary.json',
    },
    metrics=[baseline.as_row('qwen-zero-shot'), selected_dev.as_row('selected (held-out speaker)')],
    stages=[{'name': s.name, 'corpus_weights': s.corpus_weights, 'epochs': s.epochs, 'learning_rate': s.learning_rate} for s in stages],
    notes=f"Shipped passes: {[p['name'] for p in DECODE_CONFIG['passes']]}, combiner `{DECODE_CONFIG['combiner']}`, "
          f"{len(FINAL_PIPELINE.rules)} post-processing rules.",
)
print('Repository:', published.url())
print('Training metrics (TensorBoard):', published.url() + '/tensorboard')
print('Archive download:', f'https://huggingface.co/{HUB.repo_id}/resolve/main/submission.zip')